In [ ]:
import sys
import os
import importlib
import time
import datetime
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

# Add parent directory to sys.path to allow imports from sibling modules 
sys.path.append(os.path.abspath(os.path.join('..')))

# Import the database initialization module
import init_database

# Reload the module to ensure the latest changes are used
importlib.reload(init_database)

engine = init_database.init_db()

print("✅ Module init_database reloaded successfully!")

In [3]:
df_dim_products = pd.read_sql("SELECT * FROM gold.dim_products", engine)
df_dim_customers = pd.read_sql("SELECT * FROM gold.dim_customers", engine)
df_fact_sales = pd.read_sql("SELECT * FROM gold.fact_sales", engine)


In [ ]:
# Check for Uniqueness of Customer Key in gold.dim_customers
print('>> Checking dim_customers: Surrogate Key Uniqueness')

cust_key_check = df_dim_customers.groupby('customer_sk').size().reset_index(name='duplicate_count')
cust_key_check = cust_key_check[cust_key_check['duplicate_count'] > 1]

if cust_key_check.empty:
    print("✅ Pass: All customer keys are unique.")
else:
    print(f"❌ Fail: Found {len(cust_key_check)} duplicate keys.")
    display(cust_key_check)

In [ ]:
# Check for Uniqueness of Product Key in gold.dim_products
print('>> Checking dim_products: Surrogate Key Uniqueness')

prd_key_check = df_dim_products.groupby('product_sk').size().reset_index(name='duplicate_count')
prd_key_check = prd_key_check[prd_key_check['duplicate_count'] > 1]

if prd_key_check.empty:
    print("✅ Pass: All product keys are unique.")
else:
    print(f"❌ Fail: Found {len(prd_key_check)} duplicate keys.")
    display(prd_key_check)

In [ ]:
# Check Data Model Connectivity between Fact and Dimensions
print('>> Checking gold.fact_sales: Referential Integrity')

# Perform LEFT JOIN to identify missing links
integrity_check = df_fact_sales.merge(df_dim_customers[['customer_sk']], on='customer_sk', how='left', indicator='cust_link')
integrity_check = integrity_check.merge(df_dim_products[['product_sk']], on='product_sk', how='left', indicator='prd_link')

# Filter for records where the join failed (Missing in dimensions)
missing_links = integrity_check[
    (integrity_check['cust_link'] == 'left_only') | 
    (integrity_check['prd_link'] == 'left_only')
]

if missing_links.empty:
    print("✅ Pass: All sales records are correctly linked to dimensions.")
else:
    print(f"❌ Fail: Found {len(missing_links)} sales records with broken links.")
    display(missing_links)